# MIR — Fine-tuning avec Triplet Loss
### Metric learning (online hard mining) sur GPU Colab
**Ordre d'exécution :** cellules 1→2→3→4→5 puis répéter 6 pour chaque modèle.

In [ ]:
# ── Cellule 1 : Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')
# Créez le dossier MIR_Project dans votre Drive avant d'exécuter cette cellule

In [ ]:
# ── Cellule 2 : Installer les dépendances
!pip install timm==1.0.15 --quiet

In [ ]:
# ── Cellule 3 : Uploader le code du projet
# Option A : depuis Google Drive (si vous avez copié le dossier app/)
import sys, os, shutil

DRIVE_PROJECT = '/content/drive/MyDrive/MIR_Project'
COLAB_PROJECT = '/content/MIR_Project'

if os.path.exists(COLAB_PROJECT):
    shutil.rmtree(COLAB_PROJECT)
shutil.copytree(DRIVE_PROJECT, COLAB_PROJECT, ignore=shutil.ignore_patterns('indexes', '*.npz', '*.npy', 'node_modules', 'frontend', '__pycache__'))
sys.path.insert(0, COLAB_PROJECT)

print('Dataset images:', len(os.listdir(os.path.join(COLAB_PROJECT, 'dataset'))))
print('GPU:', os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip())

In [ ]:
# ── Cellule 4 : Vérifier le split train/val
from app.training.dataset import build_splits, CarDataset, TRAIN_TRANSFORMS

DATASET_DIR = os.path.join(COLAB_PROJECT, 'dataset')
SAVE_DIR    = os.path.join(DRIVE_PROJECT, 'models')   # sauvegarde sur Drive
os.makedirs(SAVE_DIR, exist_ok=True)

train_paths, val_paths = build_splits(DATASET_DIR, val_ratio=0.2)
ds = CarDataset(train_paths, TRAIN_TRANSFORMS)
print(f'Train: {len(train_paths)} | Val: {len(val_paths)} | Classes: {ds.num_classes}')

In [ ]:
# ── Cellule 5 : Vérifier le modèle et les paramètres entraînables
from app.training.models import MetricModel
import torch

for name in ['mobilenetv2', 'resnet50', 'vit_base', 'dinov2']:
    m = MetricModel(name)
    print(f'{name:15s}: {m.count_trainable():>10,} params entraînables')
    del m

In [ ]:
# ── Cellule 6 : Entraînement — modifier MODEL_NAME pour chaque modèle
from app.training.trainer import train

# ┌─────────────────────────────────────────────────────┐
# │  Changer MODEL_NAME pour chaque run :               │
# │  'mobilenetv2' | 'resnet50' | 'vit_base' | 'dinov2' │
# └─────────────────────────────────────────────────────┘
MODEL_NAME = 'mobilenetv2'

# Hyperparamètres
CONFIG = {
    'mobilenetv2': dict(num_epochs=25, batch_size=64,  lr=5e-4, margin=0.3),
    'resnet50':    dict(num_epochs=20, batch_size=48,  lr=2e-4, margin=0.3),
    'vit_base':    dict(num_epochs=15, batch_size=32,  lr=1e-4, margin=0.3),
    'dinov2':      dict(num_epochs=15, batch_size=16,  lr=5e-5, margin=0.3),
}

history, best_r1 = train(
    model_name  = MODEL_NAME,
    dataset_dir = DATASET_DIR,
    save_dir    = SAVE_DIR,
    device_str  = 'cuda',
    **CONFIG[MODEL_NAME],
)
print(f'\nTerminé. Meilleur Recall@1 = {best_r1:.3f}')
print(f'Modèle sauvé dans : {SAVE_DIR}/{MODEL_NAME}_best.pth')

In [ ]:
# ── Cellule 7 : Visualiser les courbes d'entraînement
import json, matplotlib.pyplot as plt

hist_path = os.path.join(SAVE_DIR, f'{MODEL_NAME}_history.json')
with open(hist_path) as f:
    history = json.load(f)

epochs    = [h['epoch']      for h in history]
train_l   = [h['train_loss'] for h in history]
val_l     = [h['val_loss']   for h in history]
recall1   = [h['recall@1']   for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_l, label='Train loss')
ax1.plot(epochs, val_l,   label='Val loss')
ax1.set_title(f'{MODEL_NAME} — Triplet Loss')
ax1.legend(); ax1.grid(True)

ax2.plot(epochs, recall1, color='green', label='Recall@1')
ax2.set_title(f'{MODEL_NAME} — Recall@1 (val)')
ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, f'{MODEL_NAME}_curves.png'), dpi=120)
plt.show()

In [ ]:
# ── Cellule 8 (optionnel) : Comparer tous les modèles entraînés
import os, json

models = ['mobilenetv2', 'resnet50', 'vit_base', 'dinov2']
print(f'{"Modèle":15s} | {"Best R@1":>10s} | Époques')
print('-' * 40)
for m in models:
    h_path = os.path.join(SAVE_DIR, f'{m}_history.json')
    if not os.path.exists(h_path):
        print(f'{m:15s} | non entraîné')
        continue
    with open(h_path) as f:
        hist = json.load(f)
    best = max(h['recall@1'] for h in hist)
    print(f'{m:15s} | {best:>10.3f} | {len(hist)} époques')